In [2]:
include("../RayTracing.jl")

Main.RayTracing

In [3]:
RayTracing.spectrum_from_float(0.0)

3-element Main.RayTracing.Spectrum with indices SOneTo(3):
 0.0
 0.0
 0.0

In [ ]:
function parse_file(fpath::String)
    content = read(fpath, String)

    # Remove leading/trailing whitespace and split into tokens
    tokens = split(strip(content))
    
    # Initialize return values
    nx = nothing
    ny = nothing
    nz = nothing
    p0 = nothing
    p1 = nothing
    density = nothing
    Lescale = nothing
    i = 3 # Start after command and name
    
    while i <= length(tokens)
        if tokens[i] == "\"string" && i + 2 <= length(tokens)
            param_name = strip(tokens[i+1], '"')
            i += 2
            # Handle both formats: with and without brackets
            if i <= length(tokens) && tokens[i] == "["
                i += 1
                # Read until closing bracket (but ignore the value for now)
                while i <= length(tokens) && tokens[i] != "]"
                    i += 1
                end
                if i <= length(tokens) && tokens[i] == "]"
                    i += 1
                end
            else
                # Old format without brackets
                i += 1
            end
        elseif tokens[i] == "\"integer" && i + 2 <= length(tokens)
            param_name = strip(tokens[i+1], '"')
            i += 2
            
            # Handle both formats: with and without brackets
            value = nothing
            if i <= length(tokens) && tokens[i] == "["
                i += 1
                if i <= length(tokens) && tokens[i] != "]"
                    value = RayTracing.parse(Int, tokens[i])
                    i += 1
                end
                # Skip to closing bracket
                while i <= length(tokens) && tokens[i] != "]"
                    i += 1
                end
                if i <= length(tokens) && tokens[i] == "]"
                    i += 1
                end
            else
                # Old format without brackets
                value = RayTracing.parse(Int, tokens[i])
                i += 1
            end
            
            if param_name == "nx"
                nx = value
            elseif param_name == "ny"
                ny = value
            elseif param_name == "nz"
                nz = value
            end
            
        elseif (tokens[i] == "\"point" || tokens[i] == "\"point3") && i + 1 <= length(tokens)
            param_name = strip(tokens[i+1], '"')
            i += 2
            
            # Expect opening bracket
            if i <= length(tokens) && tokens[i] == "["
                i += 1
                coords = Float64[]
                # Read coordinates until closing bracket
                while i <= length(tokens) && tokens[i] != "]"
                    push!(coords, RayTracing.parse(Float64, tokens[i]))
                    i += 1
                end
                if i <= length(tokens) && tokens[i] == "]"
                    i += 1
                end
                
                if param_name == "p0"
                    p0 = coords
                elseif param_name == "p1"
                    p1 = coords
                end
            end
            
        elseif tokens[i] == "\"float" && i + 2 <= length(tokens)
            param_name = strip(tokens[i+1], '"')
            i += 2
            
            # Expect opening bracket
            if i <= length(tokens) && tokens[i] == "["
                i += 1
                if param_name == "density"
                    # Fast parsing for large density array
                    density = parse_float_array_fast(tokens, i)
                    # Skip to after the closing bracket
                    while i <= length(tokens) && tokens[i] != "]"
                        i += 1
                    end
                    if i <= length(tokens) && tokens[i] == "]"
                        i += 1
                    end
                elseif param_name == "Lescale"
                    # Fast parsing for large Lescale array
                    Lescale = parse_float_array_fast(tokens, i)
                    # Skip to after the closing bracket
                    while i <= length(tokens) && tokens[i] != "]"
                        i += 1
                    end
                    if i <= length(tokens) && tokens[i] == "]"
                        i += 1
                    end
                else
                    # Regular float array parsing
                    values = Float64[]
                    while i <= length(tokens) && tokens[i] != "]"
                        push!(values, RayTracing.parse(Float64, tokens[i]))
                        i += 1
                    end
                    if i <= length(tokens) && tokens[i] == "]"
                        i += 1
                    end
                end
            end
        else
            # Skip unrecognized tokens
            i += 1
        end
    end
    return nx, ny, nz, p0, p1, density, Lescale
end

function parse_float_array_fast(tokens, start_idx)
    # Pre-allocate with reasonable size
    result = Float64[]
    sizehint!(result, 1_000_000) # Hint for large arrays
    i = start_idx
    while i <= length(tokens) && tokens[i] != "]"
        push!(result, RayTracing.parse(Float64, tokens[i]))
        i += 1
    end
    return result
end

parse_float_array_fast (generic function with 1 method)

In [ ]:
nx, ny, nz, p0, p1, density, le_grid = parse_file(RayTracing.jmfp("/Users/johnmyslinski/Documents/pbrt-v3-scenes/cloud/geometry/density_render.70.pbrt"))

(100, 100, 40, [0.01, 0.01, 0.01], [1.99, 1.99, 0.79], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [ ]:
nx, ny, nz, p0, p1, density, le_grid = parse_file(RayTracing.jmfp("/Users/johnmyslinski/Documents/pbrt-v4-scenes/smoke-plume/geometry/density_big_0084.pbrt"))

(192, 256, 192, [0.0, 0.0, 0.0], [0.75, 1.0, 0.75], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [ ]:
nx, ny, nz, p0, p1, density, le_grid = parse_file(RayTracing.jmfp("/Users/johnmyslinski/Documents/pbrt-v4-volumes/scenes/anemone/geometry/anemone_medium.pbrt"))